In [ ]:
import json
import uproot
import awkward as ak
import vector
import numpy as np
import matplotlib.pyplot as plt
import warnings

# plotting params
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "axes.grid": True,
        "grid.alpha": 0.6,
        "grid.linestyle": "--",
        "font.size": 14,
        "figure.dpi": 200,
    }
)

# Suppress a harmless warning from the vector library with awkward arrays
warnings.filterwarnings("ignore", message="Passing an awkward array to a ufunc")

# Register the vector library with awkward array
ak.behavior.update(vector.backends.awkward.behavior)

# --- CONFIGURATION ---
with open("hh-bbbb-obj-config.json", "r") as config_file:
    CONFIG = json.load(config_file)

with open("config_part.json", "r") as f:
    config_part = json.load(f)

In [ ]:
# Check if samples are actually distinct

n_identical = 0
print("Checking sample uniqueness...")
for i in range(len(x)):
    for j in range(i + 1, len(x)):
        diff = (x[i] - x[j]).abs().sum().item()
        if diff < 1e-3:
            print(f"Samples {i} and {j} are nearly identical! diff={diff}")
            n_identical += 1
print(f"Number of nearly identical samples: {n_identical}")

# Check sample statistics
print("\nSample-wise statistics (mean of features):")
for i in range(len(x)):
    print(
        f"Sample {i}: mean={x[i].mean().item():.4f}, std={x[i].std().item():.4f}, label={y[i].item()}"
    )

# Check how many non-zero particles per sample
print("\nNon-zero particles per sample:")
for i in range(len(x)):
    nonzero = (x[i].abs().sum(dim=-1) > 1e-6).sum().item()
    print(f"Sample {i}: {nonzero} non-zero particles out of {x.shape[1]}")

if n_identical > 0:
    print("\nDATA ISSUE: Identical samples have different labels!")
    print("Please check the dataset for consistency.")
else:
    print("\nAll samples are unique.")

In [ ]:
import os

if not os.getcwd().endswith("root-obj-perf"):
    os.chdir("root-obj-perf")
print(f"Changed working directory to: {os.getcwd()}")

import importlib
import make_dataset
import data_loading_helpers

importlib.reload(data_loading_helpers)
importlib.reload(make_dataset)

from make_dataset import generate_dataset

generate_dataset(
    config_path="hh-bbbb-obj-config.json",
    output_file="data/cls_reg_data/l1_puppi_16.npz",
    data_dir="data/hh4b/",
    collection_key="l1extpf",
    on_signal=False,
    num_constituents=16,
    higgs_mode=False,
)

In [ ]:
import matplotlib.pyplot as plt

n_const_per_jet_train_signal_pf = (
    all_masks_train_pf[all_labels_train_pf == 1].sum(dim=1).cpu().numpy()
)
n_const_per_jet_train_background_pf = (
    all_masks_train_pf[all_labels_train_pf == 0].sum(dim=1).cpu().numpy()
)
n_const_per_jet_train_signal_puppi = (
    all_masks_train_puppi[all_labels_train_puppi == 1].sum(dim=1).cpu().numpy()
)
n_const_per_jet_train_background_puppi = (
    all_masks_train_puppi[all_labels_train_puppi == 0].sum(dim=1).cpu().numpy()
)

fig, ax = plt.subplots()
ax.hist(
    n_const_per_jet_train_signal_pf,
    bins=32,
    range=(0, 32),
    density=True,
    alpha=0.5,
    label="Train PF Signal",
)
ax.hist(
    n_const_per_jet_train_background_pf,
    bins=32,
    range=(0, 32),
    density=True,
    alpha=0.5,
    label="Train PF Background",
)
ax.hist(
    n_const_per_jet_train_signal_puppi,
    bins=32,
    range=(0, 32),
    density=True,
    histtype="step",
    label="Train PUPPI Signal",
)
ax.hist(
    n_const_per_jet_train_background_puppi,
    bins=32,
    range=(0, 32),
    density=True,
    histtype="step",
    label="Train PUPPI Background",
)
ax.set_xlabel("Number of Constituents per Jet")
ax.set_ylabel("Density")
ax.set_title("Distribution of Number of Constituents per Jet: Train PF vs PUPPI")
ax.legend()
plt.show()

In [ ]:
import importlib
import train_part_data

importlib.reload(train_part_data)

from train_part_data import (
    StratifiedJetDataset,
    match_sizes_and_class_ratios,
    stratified_train_val_split,
    _compute_jet_features,
)

import json

with open("config_part.json", "r") as f:
    config_part = json.load(f)

pt_range = (40, 70)
eta_range = (-2.4, 2.4)

puppi_ds = StratifiedJetDataset(config_part["training"]["data"]["puppi_data_path"])
pf_ds = StratifiedJetDataset(config_part["training"]["data"]["pf_data_path"])
puppi_ds.apply_kinematic_filter(
    pt_min=pt_range[0], pt_max=pt_range[1], eta_min=eta_range[0], eta_max=eta_range[1]
)
pf_ds.apply_kinematic_filter(
    pt_min=pt_range[0], pt_max=pt_range[1], eta_min=eta_range[0], eta_max=eta_range[1]
)
(
    train_puppi,
    val_puppi,
    train_puppi_indices,
    val_puppi_indices,
    train_labels_puppi,
    val_labels_puppi,
) = stratified_train_val_split(puppi_ds, val_split=0.3, random_state=42, verbose=False)

(
    train_pf,
    val_pf,
    train_pf_indices,
    val_pf_indices,
    train_labels_pf,
    val_labels_pf,
) = stratified_train_val_split(pf_ds, val_split=0.3, random_state=42, verbose=False)

puppi_small_ds, pf_small_ds, puppi_labels, pf_labels = match_sizes_and_class_ratios(
    train_puppi,
    train_pf,
    puppi_ds,
    pf_ds,
    target_class1_ratio=None,
    random_state=42,
)

puppi_indices = np.array(puppi_small_ds.indices)
pf_indices = np.array(pf_small_ds.indices)

puppi_labels = puppi_ds.labels[puppi_indices]
pf_labels = pf_ds.labels[pf_indices]

puppi_bkg_indices = puppi_indices[puppi_labels == 0]
pf_bkg_indices = pf_indices[pf_labels == 0]
puppi_bkg = puppi_ds[puppi_bkg_indices]
pf_bkg = pf_ds[pf_bkg_indices]

puppi_sig_indices = puppi_indices[puppi_labels == 1]
pf_sig_indices = pf_indices[pf_labels == 1]
puppi_sig = puppi_ds[puppi_sig_indices]
pf_sig = pf_ds[pf_sig_indices]

puppi_bkg_jet_feat = _compute_jet_features(puppi_ds, puppi_bkg_indices)
puppi_sig_jet_feat = _compute_jet_features(puppi_ds, puppi_sig_indices)
pf_bkg_jet_feat = _compute_jet_features(pf_ds, pf_bkg_indices)
pf_sig_jet_feat = _compute_jet_features(pf_ds, pf_sig_indices)


n_bins_pt = 20
n_bins_eta = 20

pt_bins = np.linspace(pt_range[0], pt_range[1], n_bins_pt + 1)
eta_bins = np.linspace(eta_range[0], eta_range[1], n_bins_eta + 1)
puppi_bkg_dist, _, _ = np.histogram2d(
    puppi_bkg_jet_feat["eta"],
    puppi_bkg_jet_feat["pt"],
    bins=(eta_bins, pt_bins),
    density=False,
)
puppi_sig_dist, _, _ = np.histogram2d(
    puppi_sig_jet_feat["eta"],
    puppi_sig_jet_feat["pt"],
    bins=(eta_bins, pt_bins),
    density=False,
)
pf_bkg_dist, _, _ = np.histogram2d(
    pf_bkg_jet_feat["eta"],
    pf_bkg_jet_feat["pt"],
    bins=(eta_bins, pt_bins),
    density=False,
)
pf_sig_dist, _, _ = np.histogram2d(
    pf_sig_jet_feat["eta"],
    pf_sig_jet_feat["pt"],
    bins=(eta_bins, pt_bins),
    density=False,
)

puppi_sig_ratio = np.divide(
    puppi_bkg_dist,
    puppi_sig_dist,
    out=np.zeros_like(puppi_sig_dist),
    where=puppi_sig_dist > 0,
)
# puppi_sig_ratio = gaussian_filter(puppi_sig_ratio, smooth_sigma)
pf_sig_ratio = np.divide(
    puppi_bkg_dist, pf_sig_dist, out=np.zeros_like(pf_sig_dist), where=pf_sig_dist > 0
)
# pf_sig_ratio = gaussian_filter(pf_sig_ratio, smooth_sigma)
pf_bkg_ratio = np.divide(
    puppi_bkg_dist, pf_bkg_dist, out=np.zeros_like(pf_bkg_dist), where=pf_bkg_dist > 0
)
# pf_bkg_ratio = gaussian_filter(pf_bkg_ratio, smooth_sigma)

puppi_sig_idx = np.clip(
    np.digitize(puppi_sig_jet_feat["eta"], bins=eta_bins) - 1, 0, n_bins_eta - 1
)
puppi_sig_idy = np.clip(
    np.digitize(puppi_sig_jet_feat["pt"], bins=pt_bins) - 1, 0, n_bins_pt - 1
)
puppi_sig_weight = puppi_sig_ratio[puppi_sig_idx, puppi_sig_idy]

pf_sig_idx = np.clip(
    np.digitize(pf_sig_jet_feat["eta"], bins=eta_bins) - 1, 0, n_bins_eta - 1
)
pf_sig_idy = np.clip(
    np.digitize(pf_sig_jet_feat["pt"], bins=pt_bins) - 1, 0, n_bins_pt - 1
)
pf_sig_weight = pf_sig_ratio[pf_sig_idx, pf_sig_idy]

pf_bkg_idx = np.clip(
    np.digitize(pf_bkg_jet_feat["eta"], bins=eta_bins) - 1, 0, n_bins_eta - 1
)
pf_bkg_idy = np.clip(
    np.digitize(pf_bkg_jet_feat["pt"], bins=pt_bins) - 1, 0, n_bins_pt - 1
)
pf_bkg_weight = pf_bkg_ratio[pf_bkg_idx, pf_bkg_idy]

puppi_bkg_weights = np.ones_like(puppi_bkg_jet_feat["pt"])


import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
ax = axes[0, 0]
ax.hist(
    pf_sig_jet_feat["pt"],
    bins=n_bins_pt,
    range=pt_range,
    density=False,
    alpha=0.5,
    label="PF Sig",
)
ax.hist(
    pf_bkg_jet_feat["pt"],
    bins=n_bins_pt,
    range=pt_range,
    density=False,
    alpha=0.5,
    label="PF Bkg",
)
ax.hist(
    puppi_sig_jet_feat["pt"],
    bins=n_bins_pt,
    range=pt_range,
    density=False,
    histtype="step",
    label="PUPPI Sig",
)
ax.hist(
    puppi_bkg_jet_feat["pt"],
    bins=n_bins_pt,
    range=pt_range,
    density=False,
    histtype="step",
    label="PUPPI Bkg",
)
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Unweighted $p_T$")
ax.legend()
ax = axes[0, 1]
ax.hist(
    pf_sig_jet_feat["eta"],
    bins=n_bins_eta,
    range=eta_range,
    density=False,
    alpha=0.5,
    label="PF Sig",
)
ax.hist(
    pf_bkg_jet_feat["eta"],
    bins=n_bins_eta,
    range=eta_range,
    density=False,
    alpha=0.5,
    label="PF Bkg",
)
ax.hist(
    puppi_sig_jet_feat["eta"],
    bins=n_bins_eta,
    range=eta_range,
    density=False,
    histtype="step",
    label="PUPPI Sig",
)
ax.hist(
    puppi_bkg_jet_feat["eta"],
    bins=n_bins_eta,
    range=eta_range,
    density=False,
    histtype="step",
    label="PUPPI Bkg",
)
ax.set_xlabel("Jet $\\eta$")
ax.set_ylabel("Density")
ax.set_title("Unweighted $\\eta$")
ax.legend()
ax = axes[1, 0]
ax.hist(
    pf_sig_jet_feat["pt"],
    bins=n_bins_pt,
    range=pt_range,
    density=False,
    alpha=0.5,
    label="PF Sig",
    weights=pf_sig_weight.flatten(),
)
ax.hist(
    pf_bkg_jet_feat["pt"],
    bins=n_bins_pt,
    range=pt_range,
    density=False,
    alpha=0.5,
    label="PF Bkg",
    weights=pf_bkg_weight.flatten(),
)
ax.hist(
    puppi_sig_jet_feat["pt"],
    bins=n_bins_pt,
    range=pt_range,
    density=False,
    histtype="step",
    label="PUPPI Sig",
    weights=puppi_sig_weight.flatten(),
)
ax.hist(
    puppi_bkg_jet_feat["pt"],
    bins=n_bins_pt,
    range=pt_range,
    density=False,
    histtype="step",
    label="PUPPI Bkg",
    weights=puppi_bkg_weights.flatten(),
)
ax.set_xlabel("Jet $p_T$ [GeV]")
ax.set_ylabel("Density")
ax.set_title("Reweighted $p_T$ (to PUPPI bkg)")
ax.legend()

ax = axes[1, 1]
ax.hist(
    pf_sig_jet_feat["eta"],
    bins=n_bins_eta,
    range=eta_range,
    density=False,
    alpha=0.5,
    label="PF Sig",
    weights=pf_sig_weight.flatten(),
)
ax.hist(
    pf_bkg_jet_feat["eta"],
    bins=n_bins_eta,
    range=eta_range,
    density=False,
    alpha=0.5,
    label="PF Bkg",
    weights=pf_bkg_weight.flatten(),
)
ax.hist(
    puppi_sig_jet_feat["eta"],
    bins=n_bins_eta,
    range=eta_range,
    density=False,
    histtype="step",
    label="PUPPI Sig",
    weights=puppi_sig_weight.flatten(),
)
ax.hist(
    puppi_bkg_jet_feat["eta"],
    bins=n_bins_eta,
    range=eta_range,
    density=False,
    histtype="step",
    label="PUPPI Bkg",
)
ax.set_xlabel("Jet $\\eta$")
ax.set_ylabel("Density")
ax.set_title("Reweighted $\\eta$ (to PUPPI bkg)")
ax.legend()
# fig.colorbar(im, ax=ax)
# plt.suptitle("Mode 6 Weighting Verification: Reweighting Maps and Resulting Distributions", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# === Constituent-level relative feature distributions (before & after reweighting) ===

fig, axes = plt.subplots(2, 4, figsize=(28, 10))

rel_feats = [
    ("delta_m", "$\\Delta m$ (constituent − jet) [GeV]", None, 50),
    ("rel_pt", "Relative $p_T$ (constituent / jet)", (0, 1), 50),
    ("delta_eta", "$\\Delta\\eta$ (constituent − jet)", (-0.4, 0.4), 50),
    ("delta_phi", "$\\Delta\\phi$ (constituent − jet)", (-0.4, 0.4), 50),
]

for col, (feat_key, xlabel, rng, nbins) in enumerate(rel_feats):
    # Get flattened constituent-level values + weights (masked)
    flat_vals = {}
    flat_wts = {}
    for k, (ds, idx) in groups.items():
        mask_np = ds.mask[idx].numpy().astype(bool)  # (N, n_const)
        vals = feats[k][feat_key]  # (N, n_const)
        w_jet = weights[k]  # (N,)
        # broadcast jet weight to constituents, then flatten with mask
        w_const = np.broadcast_to(w_jet[:, None], vals.shape)  # (N, n_const)
        flat_vals[k] = vals[mask_np]
        flat_wts[k] = w_const[mask_np]

    # Row 0: Unweighted
    ax = axes[0, col]
    for k, style in [
        ("PF Sig", dict(alpha=0.5)),
        ("PF Bkg", dict(alpha=0.5)),
        ("PUPPI Sig", dict(histtype="step")),
        ("PUPPI Bkg", dict(histtype="step")),
    ]:
        ax.hist(flat_vals[k], bins=nbins, range=rng, density=True, label=k, **style)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(f"Unweighted {feat_key}")
    ax.legend(fontsize=8)

    # Row 1: Reweighted
    ax = axes[1, col]
    for k, style in [
        ("PF Sig", dict(alpha=0.5)),
        ("PF Bkg", dict(alpha=0.5)),
        ("PUPPI Sig", dict(histtype="step")),
        ("PUPPI Bkg", dict(histtype="step")),
    ]:
        ax.hist(
            flat_vals[k],
            bins=nbins,
            range=rng,
            density=True,
            weights=flat_wts[k],
            label=k,
            **style,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Weighted density")
    ax.set_title(f"Reweighted {feat_key}")
    ax.legend(fontsize=8)

plt.suptitle(
    "Mode 9: Constituent-level relative features (before & after reweighting)",
    fontsize=14,
    y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# === Jet-level feature distributions (before & after reweighting) ===

fig, axes = plt.subplots(2, 4, figsize=(28, 10))

jet_feats = [
    ("jet_pt", "Jet $p_T$ [GeV]", pt_range, 30),
    ("jet_eta", "Jet $\\eta$", eta_range, 30),
    ("mean_rel_pt", "Mean rel $p_T$", None, 30),
    ("mean_delta_eta", "Mean $\\Delta\\eta$", None, 30),
]

for col, (feat_key, xlabel, rng, nbins) in enumerate(jet_feats):
    # Row 0: Unweighted
    ax = axes[0, col]
    for k, style in [
        ("PF Sig", dict(alpha=0.5)),
        ("PF Bkg", dict(alpha=0.5)),
        ("PUPPI Sig", dict(histtype="step")),
        ("PUPPI Bkg", dict(histtype="step")),
    ]:
        ax.hist(
            feats[k][feat_key], bins=nbins, range=rng, density=True, label=k, **style
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(f"Unweighted {feat_key}")
    ax.legend(fontsize=8)

    # Row 1: Reweighted
    ax = axes[1, col]
    for k, style in [
        ("PF Sig", dict(alpha=0.5)),
        ("PF Bkg", dict(alpha=0.5)),
        ("PUPPI Sig", dict(histtype="step")),
        ("PUPPI Bkg", dict(histtype="step")),
    ]:
        ax.hist(
            feats[k][feat_key],
            bins=nbins,
            range=rng,
            density=True,
            weights=weights[k],
            label=k,
            **style,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Weighted density")
    ax.set_title(f"Reweighted {feat_key}")
    ax.legend(fontsize=8)

plt.suptitle(
    "Mode 9: Jet-level features (before & after reweighting)", fontsize=14, y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
import importlib
import train_part

importlib.reload(train_part)

from train_part import run_training

run_training("config_part.json")

In [ ]:
import h5py
import time
import numpy as np

# Replace with your file path and dataset name
with h5py.File("data/qcd_bkg_cls_reg/l1_puppi_16_flat.h5", "r") as f:
    print(f.keys())
    dset = f["jet_pt"]
    n_samples = len(dset)
    indices = np.random.choice(n_samples, 1000, replace=False)

    # Test 1: Random Access (What your Dataloader does)
    start = time.time()
    for idx in indices:
        _ = dset[idx]
    print(f"Random Access: {time.time() - start:.4f}s")

    # Test 2: Sequential Access (HDF5's "Happy Path")
    start = time.time()
    _ = dset[0:1000]
    print(f"Sequential Access: {time.time() - start:.4f}s")

In [ ]:
import os

if not os.getcwd().endswith("root-obj-perf"):
    os.chdir("root-obj-perf")
print(f"Current working directory: {os.getcwd()}")

import time
import torch
from torch.utils.data import DataLoader

# Assuming you added the class to your file, import it:
from train_part_data import StratifiedJetDataset, WeakShuffleSampler

# --- CONFIGURATION ---
# Replace with the actual path to one of your .h5 datasets
DATA_PATH = "/Users/adityatandon/Documents/College/Physics/Year 4/Neural SBI/root-obj-perf/data/qcd_bkg_cls_reg/l1_puppi_16_flat.h5"
BATCH_SIZE = 128
NUM_WORKERS = 2  # Match your 4-core setup
BATCHES_TO_TEST = 50  # Set to None to test a full epoch

print("Loading dataset...")
dataset = StratifiedJetDataset(DATA_PATH, mode=None)

# --- 1. OLD METHOD (Standard Random Shuffle) ---
print("\nInitializing OLD DataLoader (Standard Shuffle)...")
old_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,  # The culprit
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

# --- 2. NEW METHOD (Weak Shuffle) ---
print("Initializing NEW DataLoader (Weak Shuffle Sampler)...")
new_sampler = WeakShuffleSampler(dataset, chunk_size=BATCH_SIZE)
new_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Must be False when using a sampler
    sampler=new_sampler,  # Using our custom sampler
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)


# --- BENCHMARK FUNCTION ---
def benchmark_loader(loader, name, num_batches=None):
    print(f"\n--- Starting benchmark: {name} ---")
    start_time = time.time()

    count = 0
    # Iterate through the dataloader
    for i, batch in enumerate(loader):
        count += 1
        # Print progress every 10 batches so you know it hasn't crashed
        if count % 10 == 0:
            print(f"  Loaded {count} batches...")

        if num_batches is not None and count >= num_batches:
            break

    end_time = time.time()
    elapsed = end_time - start_time

    print(f"✅ [{name}] Total time for {count} batches: {elapsed:.2f} seconds")
    print(f"⏱️ [{name}] Average time per batch: {(elapsed/count):.4f} seconds")
    return elapsed


# --- RUN THE TEST ---
# Run the NEW method first (so you get quick feedback that it works)
time_new = benchmark_loader(
    new_loader, "NEW Method (Weak Shuffle)", num_batches=BATCHES_TO_TEST
)

# Run the OLD method
time_old = benchmark_loader(
    old_loader, "OLD Method (Standard Shuffle)", num_batches=BATCHES_TO_TEST
)

# --- RESULTS ---
speedup = time_old / time_new
print("\n" + "=" * 50)
print("🏆 BENCHMARK RESULTS")
print("=" * 50)
print(f"Old Method Time : {time_old:.2f}s")
print(f"New Method Time : {time_new:.2f}s")
print(f"Performance Gain: The new method is {speedup:.1f}x faster!")
print("=" * 50)

Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-30To50/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
  qcd_pt_30_50: 20 chunks, 100000 events loaded, 0 total reconstructed so far

--- QCD bin: qcd_pt_50_80  (weight=1.749e+07) ---
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-50To80/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
  qcd_pt_50_80: 20 chunks, 100000 events loaded, 0 total reconstructed so far

--- QCD bin: qcd_pt_80_120  (weight=2.657e+06) ---
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-80To120/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
  qcd_pt_80_120: 20 chunks, 100000 events loaded, 1 total reconstructed so far

--- QCD bin: qcd_pt_120_170  (weight=4.678e+05) ---
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 4000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-120To170/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 0 events.
  qcd_pt_120_170: 20 chunks, 99000 events loaded, 3 total reconstructed so far

--- QCD bin: qcd_pt_170_300  (weight=1.203e+05) ---
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 4000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-170To300/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 0 events.
  qcd_pt_170_300: 20 chunks, 99000 events loaded, 21 total reconstructed so far

--- QCD bin: qcd_pt_300_470  (weight=8.157e+03) ---
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 4000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-300To470/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 0 events.
  qcd_pt_300_470: 20 chunks, 99000 events loaded, 49 total reconstructed so far

--- QCD bin: qcd_pt_470_600  (weight=6.831e+02) ---
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 2000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-470To600/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 0 events.
  qcd_pt_470_600: 20 chunks, 97000 events loaded, 91 total reconstructed so far

--- QCD bin: qcd_pt_600_inf  (weight=2.410e+02) ---
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 5000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 3000 events.

Applying custom pT cut of 0.0 GeV for l1extpuppi jets...
Loading data from /rds/general/user/at3722/home/root-obj-perf/data/QCD_Pt-600ToInf/data_*.root...
Reshaping data into nested objects...
Creating 4-vector objects...
Loaded and restructured 0 events.
  qcd_pt_600_inf: 19 chunks, 93000 events loaded, 152 total reconstructed so far

Total QCD: 152 reconstructed events from 887000 processed
QCD weights summary: min=2.4e+02, max=2.7e+06, sum=6.030e+06
QCD mHH: weighted mean=582.6, unweighted median=945.7 GeV

============================================================
Di-Higgs reconstruction complete
  Collection: l1extpuppi (puppi)
  Working point: loose (threshold=0.5584)
  Selected: 1139 signal events with >=4 tagged jets = 2.28% of 50000 total HH events
  Signal (all 4 pure): 514 events (45.1%)
  QCD background: 152 events (from 887000 QCD events processed)
  pT regression: applied
  Pairing eff: 56.03% | Swapped: 17.32% | Total: 73.35%
============================================================


Pairing Efficiency: 61.78%
Pairing Efficiency with pairs swapped: 16.12%
Total Pairing Efficiency (including swapped): 77.89%

============================================================
Processing QCD BACKGROUND...
============================================================

--- QCD bin: qcd_pt_20_30  (weight=4.329e+08) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 2/100
  → 2 QCD events with >=4 b-tagged jets reconstructed

--- QCD bin: qcd_pt_30_50  (weight=1.172e+08) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 79/1000
  → 79 QCD events with >=4 b-tagged jets reconstructed

--- QCD bin: qcd_pt_50_80  (weight=1.749e+07) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 10/100
  → 10 QCD events with >=4 b-tagged jets reconstructed

--- QCD bin: qcd_pt_80_120  (weight=2.657e+06) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 11/100
  → 11 QCD events with >=4 b-tagged jets reconstructed

--- QCD bin: qcd_pt_120_170  (weight=4.678e+05) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 911/10000
  → 911 QCD events with >=4 b-tagged jets reconstructed

--- QCD bin: qcd_pt_170_300  (weight=1.203e+05) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 975/10000
  → 975 QCD events with >=4 b-tagged jets reconstructed

--- QCD bin: qcd_pt_300_470  (weight=8.157e+03) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 10286/99000
  → 10286 QCD events with >=4 b-tagged jets reconstructed

--- QCD bin: qcd_pt_470_600  (weight=6.831e+02) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 10931/97000
  → 10931 QCD events with >=4 b-tagged jets reconstructed

--- QCD bin: qcd_pt_600_inf  (weight=2.410e+02) ---
  Processing in chunks of 5000 events

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...

Applying custom pT cut of 0.0 GeV for l1extpf jets...
  Events with >=4 tagged jets: 11285/93000
  → 11285 QCD events with >=4 b-tagged jets reconstructed

Total QCD: 34490 reconstructed events from 310300 processed
QCD weights summary: min=2.4e+02, max=4.3e+08, sum=1.097e+10
QCD mHH: weighted mean=213.4, unweighted median=287.1 GeV

============================================================
Di-Higgs reconstruction complete
  Collection: l1extpf (pf)
  Working point: loose (threshold=0.0711)
  Selected: 17405 signal events with >=4 tagged jets = 34.81% of 50000 total HH events
  Signal (all 4 pure): 484 events (2.8%)
  QCD background: 34490 events (from 310300 QCD events processed)
  pT regression: applied
  Pairing eff: 61.78% | Swapped: 16.12% | Total: 77.89%
============================================================

Skipping Top-N jet purity efficiency plots (not needed).
  Saved: dihiggs_mass_1d_loose.png
  Saved: dihiggs_mass_2d_loose.png

======================================================================
Tagger                              S    B (wtd)   S/√(S+B)  |   S (mHH)    B (mHH)   Sig(mHH)  |   S (rHH)    B (rHH)   Sig(rHH)  |   S (both)    B (both)  Sig(both)
                                (all)      (all)      (all)  |  (250, 550) (250, 550)     window  |   R_HH<55    R_HH<55     region  |  mHH & rHH   mHH & rHH   combined
------------------------------------------------------------------------------------------------------------------------------------------------------
Trained ParT (loose)              484    1.1e+10     0.0046  |       374    2.2e+09     0.0080  |       396    3.1e+09     0.0071  |        308     1.2e+09     0.0088

  Note: B counts are cross-section-weighted sums of QCD events.
  Unweighted QCD events: 34490
======================================================================================================================================================

Working point: loose (threshold=0.0711)
pT regression: applied
Collection: l1extpf

All di-Higgs plots saved to: ../Updates/plots_pf_qcd_qreg_pos_weight_10.0/periodic_model:v1627

ROC + di-Higgs mass-only mode complete. Skipping remaining analyses.
